<a href="https://colab.research.google.com/github/gongyu-wang/IDX-Exchange-project-work/blob/main/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 - Data Preprocessing Deliverable

**Business objective:** prepare CRMLS SingleFamilyResidence sold data for ClosePrice prediction so pricing, listing strategy, and demand allocation decisions can be evaluated on the newest market month.

This notebook combines the team's Week 3 work, incorporating robustness and data leakage prevention:

1.  missing-value handling
2.  categorical-to-numeric encoding
3.  numerical issue flags and scaling
4.  time-based train/validation/test split
5.  new feature engineering: `PropertyAge`, cyclical month features, and ZIP-level median price per square foot aggregates.

## Assumptions and Review Standard

- Scope is restricted to `PropertyType == "Residential"` and `PropertySubType == "SingleFamilyResidence"`.
- Target is `ClosePrice`.
- `CloseDate` is the transaction date used for the time-based split.
- The most recent available close month is the test set (`2026-05`).
- The month immediately preceding the test set (`2026-04`) is the validation set.
- The default training window is `X = 12` immediately preceding months (`2025-04` through `2026-03`). `X` is tunable; candidate row counts are reported below.
- Missing target or close date cannot support the prediction task, so those rows are removed.
- Missing latitude/longitude is removed because the project requires geographic segmentation and enrichment.
- Numerical hard failures are removed; review-level outliers are retained with flags.
- Scaling and imputation parameters are fit on train only, then applied to validation and test to avoid leakage.

In [ ]:
import os
import time
from datetime import datetime
from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Imports and Paths

In [ ]:
# The project data covers Jan 2025 through May 2026.
# We list the months explicitly so it is obvious which files are expected.
MONTHS = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512",
    "202601", "202602", "202603", "202604", "202605",
]

# Week 3 requires the most recent month as test and the previous X months as train.
# X is tunable. We use 12 as the default and also report other candidate windows.
TRAINING_WINDOW_MONTHS = 12
TRAINING_WINDOW_CANDIDATES = [3, 6, 9, 12, 15, 16]

# Prediction target and date field used for time-based evaluation.
TARGET_COL = "ClosePrice"
DATE_COL = "CloseDate"

# Try common Colab Drive paths first, then local project paths.
# If the team runs this in Colab, the IDX Exchange path from the screenshots should work.
candidate_paths = [
    Path("/content/drive/MyDrive/IDX Exchange/Data"),
]

DATA_PATH = None
for path in candidate_paths:
    if path.exists() and list(path.glob(f"CRMLSSold{MONTHS[0]}*.csv")):
        DATA_PATH = path
        break

# Stop early if the raw monthly files cannot be found.
# This is better than silently running on the wrong folder.
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find CRMLSSold monthly CSVs. Update DATA_PATH to the folder containing CRMLSSold202501.csv ... CRMLSSold202605.csv."
    )

# Output folder for all Week 3 deliverables.
OUTPUT_DIR = Path("/content/drive/MyDrive/IDX Exchange/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_PATH:", DATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_PATH: /content/drive/MyDrive/IDX Exchange/Data
OUTPUT_DIR: /content/drive/MyDrive/IDX Exchange/outputs


## 2. Load Monthly Sold Files and Apply CRMLS Scope Filter

In [ ]:
# Each monthly CSV is loaded separately, then combined into one table.
# source_month records which file each row came from, which helps audit duplicates and time coverage.
frames = []
loaded_files = []
missing_files = []

for month in MONTHS:
    matches = sorted(DATA_PATH.glob(f"CRMLSSold{month}*.csv"))

    # If a month is missing, record it instead of failing immediately.
    # The notebook prints missing months so the team can decide if the data pull is incomplete.
    if not matches:
        missing_files.append(month)
        continue

    file_path = matches[0]
    temp = pd.read_csv(file_path, low_memory=False)
    temp["source_month"] = month
    frames.append(temp)
    loaded_files.append(file_path.name)

if not frames:
    raise ValueError("No monthly CRMLS sold files were loaded.")

# Combine all monthly data. sort=False preserves original columns without forcing alphabetical order.
df_raw = pd.concat(frames, ignore_index=True, sort=False)

print(f"Loaded {len(loaded_files)} files")
print("Missing months:", missing_files if missing_files else "none")
print("Raw shape:", df_raw.shape)

required = {"PropertyType", "PropertySubType"}
missing = required.difference(df_raw.columns)
if missing:
    raise KeyError(f"Missing required CRMLS filter columns: {missing}")

# Non-negotiable project scope:
# only Residential + SingleFamilyResidence records are valid for this pipeline.
initial_len = len(df_raw)
df = df_raw[
    (df_raw["PropertyType"].astype(str).str.strip() == "Residential")
    & (df_raw["PropertySubType"].astype(str).str.strip() == "SingleFamilyResidence")
].copy()

removed_count = initial_len - len(df)
removed_pct = (removed_count / initial_len) * 100 if initial_len > 0 else 0

print("Rows before filter:", initial_len)
print("Rows after Residential + SingleFamilyResidence filter:", len(df))
print(f"Rows removed by property filter: {removed_count} ({removed_pct:.2f}%)")
df.head()

Loaded 17 files
Missing months: none
Raw shape: (342747, 81)
Rows before filter: 342747
Rows after Residential + SingleFamilyResidence filter: 170660
Rows removed by property filter: 172087 (50.21%)


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,source_month
11,SanDiego,SanDiego,"Carpet,Tile",False,NaN,NaN,False,530000.0,1104130366,tom@structuresd.com,...,False,0.0,NaN,92021,NaN,13800.0,NaN,False,False,202501
12,SanDiego,SanDiego,NaN,False,NaN,NaN,False,700000.0,1104128744,chadtrower@gmail.com,...,False,0.0,NaN,92124,170.0,3000.0,NaN,False,False,202501
13,SanDiego,SanDiego,"Carpet,Tile",False,NaN,NaN,False,1680000.0,1104128653,lisa@premiercoastalrealty.com,...,False,2.0,NaN,92130,26.0,NaN,NaN,False,False,202501
14,CaliforniaDesert,CaliforniaDesert,Tile,True,NaN,NaN,True,1195000.0,1104128077,lisa@traditionproperties.net,...,False,3.0,NaN,92253,275.0,11761.0,NaN,False,False,202501
16,BeverlyHillsGreaterLa,BeverlyHillsGreaterLa,NaN,True,NaN,NaN,NaN,3800000.0,1104126716,Djacobson.re@gmail.com,...,False,NaN,NaN,91423,NaN,15365.0,NaN,False,False,202501


## 3. Feature Configuration

In [ ]:
# Numerical fields that are relevant for pricing or data-quality review.
# ClosePrice is included for summary checks but excluded from feature scaling later.
KEY_NUMERIC_COLS = [
    "ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "PropertyAge", "GarageSpaces", "ParkingTotal",
    "Latitude", "Longitude", "AssociationFee", "FireplacesTotal", "Stories",
    "zip_median_price_per_sqft", # New engineered feature
    "month_sin", "month_cos" # Cyclical month features
]

# Binary listing attributes. These are amenity-like fields where values should become 0/1.
BINARY_COLS = [
    "ViewYN", "WaterfrontYN", "BasementYN", "PoolPrivateYN", "AttachedGarageYN",
    "FireplaceYN", "NewConstructionYN", "latfilled", "lonfilled",
]

# Low-cardinality categorical fields can be one-hot encoded without creating too many columns.
LOW_CARDINALITY_CATEGORICAL_COLS = [
    "Levels", "AssociationFeeFrequency", "StateOrProvince",
]

# Higher-cardinality geographic fields are frequency encoded later.
# This keeps useful market-location signal without creating hundreds of dummy columns.
FREQUENCY_CATEGORICAL_COLS = [
    "City", "PostalCode", "CountyOrParish", "MLSAreaMajor",
    "HighSchoolDistrict", "ElementarySchoolDistrict", "MiddleOrJuniorSchoolDistrict",
]

# These fields are identifiers, names, addresses, or agent/office fields.
# They are kept in the audit data but should not be treated as stable model features.
DROP_FROM_MODEL = [
    "ListingKey", "ListingKeyNumeric", "ListingId", "UnparsedAddress", "ListAgentEmail",
    "ListAgentFirstName", "ListAgentLastName", "ListAgentFullName",
    "CoListAgentFirstName", "CoListAgentLastName",
    "BuyerAgentFirstName", "BuyerAgentLastName", "BuyerAgentMlsId",
    "ListOfficeName", "BuyerOfficeName", "CoListOfficeName",
    "BuilderName", "SubdivisionName", "StreetNumberNumeric",
    "ListPrice", "OriginalListPrice", # Excluded as requested
    "DaysOnMarket", # Excluded as requested
    "ContractStatusChangeDate", "PurchaseContractDate", # Excluded as closing/settlement fields
    'BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'CloseDate',
    'PropertyType', 'MlsStatus', 'ElementarySchool', 'PropertySubType',
    'BuyerOfficeAOR', 'CoBuyerAgentFirstName', 'ListingContractDate',
    'MiddleOrJuniorSchool', 'HighSchool', 'LotSizeDimensions',
    'close_month']

## 4. Date Conversion, PropertyAge, Required-Field Checks, and Duplicate Handling

This section performs critical date conversions (transforming date columns to `datetime` format). It then calculates `PropertyAge` based on `CloseDate` and `YearBuilt`, and extracts cyclical monthly features (`month_sin` and `month_cos`) from `CloseDate`. It also checks for required fields (such as `ClosePrice` and `CloseDate`) and handles duplicate listings to ensure data accuracy and consistency. These steps generate the `df_clean` DataFrame, laying the groundwork for subsequent data cleaning and feature engineering.

In [ ]:
DATE_COLS = ["CloseDate", "ContractStatusChangeDate", "PurchaseContractDate", "ListingContractDate"]

# Convert date-like columns to datetime.
# errors="coerce" changes invalid date strings to NaT, making bad dates visible in missing checks.
for col in DATE_COLS:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# ClosePrice and CloseDate are required:
# - ClosePrice is the supervised learning target.
# - CloseDate is required for the time-based train/test split.
required_for_model = [TARGET_COL, DATE_COL]
for col in required_for_model:
    if col not in df.columns:
        raise KeyError(f"Missing required modeling column: {col}")

# Remove rows that cannot support the modeling task.
# This is a justified removal, not a blind drop.
initial_len = len(df)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df[df[TARGET_COL].notna() & df[DATE_COL].notna()].copy()

removed_count = initial_len - len(df)
removed_pct = (removed_count / initial_len) * 100 if initial_len > 0 else 0
print(f"Rows removed for missing target or CloseDate: {removed_count} ({removed_pct:.2f}%)")

# Deduplicate listings.
# If ListingKey exists, it is the best available listing identifier.
# We keep the latest record by CloseDate/source_month because it is most likely to reflect final sale information.
before_dedup = len(df)
if "ListingKey" in df.columns:
    df = df.sort_values([DATE_COL, "source_month"], na_position="first")
    duplicate_count = df.duplicated("ListingKey", keep="last").sum()
    df = df.drop_duplicates("ListingKey", keep="last").copy()
    dedup_removed_pct = (duplicate_count / before_dedup) * 100 if before_dedup > 0 else 0
    print(f"Duplicate ListingKey rows removed: {int(duplicate_count)} ({dedup_removed_pct:.2f}%)")
else:
    # Fallback only if ListingKey is unavailable.
    duplicate_count = df.duplicated().sum()
    df = df.drop_duplicates().copy()
    dedup_removed_pct = (duplicate_count / before_dedup) * 100 if before_dedup > 0 else 0
    print(f"Exact duplicate rows removed: {int(duplicate_count)} ({dedup_removed_pct:.2f}%)")

# Create df_clean as a copy after initial date conversion and deduplication.
df_clean = df.copy()

print("Rows after de-duplication:", len(df_clean), "| removed total:", before_dedup - len(df_clean))
df_clean[[DATE_COL, TARGET_COL, "source_month"]].head()

Rows removed for missing target or CloseDate: 0 (0.00%)
Duplicate ListingKey rows removed: 0 (0.00%)
Rows after de-duplication: 170553 | removed total: 0


,CloseDate,ClosePrice,source_month
5881,2025-01-01,3720000.0,202501
8435,2025-01-01,400000.0,202501
8721,2025-01-01,660595.0,202501
9488,2025-01-01,1325000.0,202501
10477,2025-01-01,980000.0,202501


In [ ]:
# Calculate PropertyAge (SaleYear - YearBuilt)
df_clean["PropertyAge"] = (df_clean[DATE_COL].dt.year - pd.to_numeric(df_clean["YearBuilt"]))

# Cyclical features for month
df_clean["month_sin"] = np.sin(2 * np.pi * df_clean[DATE_COL].dt.month / 12)
df_clean["month_cos"] = np.cos(2 * np.pi * df_clean[DATE_COL].dt.month / 12)

print("PropertyAge min/max:", df_clean["PropertyAge"].min(), df_clean["PropertyAge"].max())
print("month_sin min/max:", df_clean["month_sin"].min(), df_clean["month_sin"].max())
print("month_cos min/max:", df_clean["month_cos"].min(), df_clean["month_cos"].max())

PropertyAge min/max: -1.0 249.0
month_sin min/max: -1.0 1.0
month_cos min/max: -1.0 1.0


## 5. Missing-Value and Numerical Summaries

In [ ]:
# Missing-value summary for all columns.
# This tells the team which fields need imputation, removal, or exclusion.
missing_summary = (
    pd.DataFrame({
        "missing_count": df_clean.isna().sum(),
        "missing_percent": df_clean.isna().mean() * 100,
        "dtype": df_clean.dtypes.astype(str),
    })
    .sort_values(["missing_percent", "missing_count"], ascending=False)
)

# Numerical summary for all numeric columns.
# Percentiles help identify impossible values and extreme outliers.
numerical_summary_rows = []
for col in df_clean.select_dtypes(include=[np.number]).columns:
    s = pd.to_numeric(df_clean[col], errors="coerce")
    q = s.quantile([0, .01, .05, .25, .5, .75, .95, .99, 1]).to_dict()
    numerical_summary_rows.append({
        "column": col,
        "count": int(s.count()),
        "missing_count": int(s.isna().sum()),
        "missing_pct": round(float(s.isna().mean() * 100), 4),
        "zero_count": int((s == 0).sum()),
        "negative_count": int((s < 0).sum()),
        "mean": s.mean(),
        "std": s.std(),
        "min": q.get(0),
        "p1": q.get(.01),
        "p5": q.get(.05),
        "p25": q.get(.25),
        "median": q.get(.5),
        "p75": q.get(.75),
        "p95": q.get(.95),
        "p99": q.get(.99),
        "max": q.get(1),
    })

numerical_summary = pd.DataFrame(numerical_summary_rows)

# Save summaries so the cleaning decisions are documented outside the notebook too.
missing_summary.to_csv(OUTPUT_DIR / "missing_summary.csv")
numerical_summary.to_csv(OUTPUT_DIR / "numerical_summary.csv", index=False)

display(missing_summary.head(20))
display(numerical_summary[numerical_summary["column"].isin(KEY_NUMERIC_COLS)])

,missing_count,missing_percent,dtype
FireplacesTotal,170132,100.000000,float64
AboveGradeFinishedArea,170132,100.000000,float64
TaxAnnualAmount,170132,100.000000,float64
TaxYear,170132,100.000000,float64
ElementarySchoolDistrict,170132,100.000000,float64
BusinessType,170132,100.000000,object
CoveredSpaces,170132,100.000000,float64
MiddleOrJuniorSchoolDistrict,170132,100.000000,float64
BelowGradeFinishedArea,168967,99.315238,float64
BuilderName,162378,95.442362,object


,column,count,missing_count,missing_pct,zero_count,negative_count,mean,std,min,p1,p5,p25,median,p75,p95,p99,max
6,ClosePrice,170132,0,0.0000,0,0,1.323586e+06,6.951611e+06,1.750000,239155.000000,370000.000000,630000.000000,8.998000e+05,1.430000e+06,3.175000e+06,6.334140e+06,9.895000e+08
7,Latitude,170132,0,0.0000,0,0,3.474099e+01,1.698812e+00,32.060512,32.672395,32.844675,33.758643,3.408073e+01,3.485677e+01,3.795926e+01,3.973051e+01,4.197154e+01
8,Longitude,170132,0,0.0000,0,170132,-1.186474e+02,1.852850e+00,-124.193201,-122.440763,-122.189732,-119.170835,-1.180302e+02,-1.172613e+02,-1.165886e+02,-1.162441e+02,-1.143114e+02
9,LivingArea,170041,91,0.0535,0,0,2.039315e+03,1.014995e+03,2.000000,744.000000,969.000000,1383.000000,1.811000e+03,2.430000e+03,3.810000e+03,5.600000e+03,2.000000e+04
12,FireplacesTotal,0,170132,100.0000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,ParkingTotal,170131,1,0.0006,9830,27,3.084926e+00,4.740231e+01,-143.000000,0.000000,0.000000,2.000000,2.000000e+00,3.000000e+00,6.000000e+00,1.100000e+01,1.572000e+04
19,YearBuilt,170037,95,0.0558,0,0,1.975615e+03,2.749389e+01,1801.000000,1910.000000,1926.000000,1956.000000,1.976000e+03,1.998000e+03,2.022000e+03,2.025000e+03,2.026000e+03
21,BathroomsTotalInteger,170113,19,0.0112,52,0,2.622857e+00,1.109512e+00,0.000000,1.000000,1.000000,2.000000,2.000000e+00,3.000000e+00,5.000000e+00,6.000000e+00,1.500000e+01
24,BedroomsTotal,170132,0,0.0000,80,0,3.486034e+00,9.526833e-01,0.000000,2.000000,2.000000,3.000000,3.000000e+00,4.000000e+00,5.000000e+00,6.000000e+00,1.500000e+01
29,Stories,151609,18523,10.8874,0,0,1.349973e+00,4.769626e-01,1.000000,1.000000,1.000000,1.000000,1.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00


## 6. Missing Values, Binary Encoding, and Numerical Quality Flags

Decision rule:

- Binary Y/N fields are converted to 0/1; missing is treated as 0 only for amenity-presence fields where missing means the listing did not report the feature as present.
- Hard numerical failures are removed because they corrupt model learning and stakeholder price interpretation.
- Review outliers are retained with flags because luxury, rural, or unusual properties can be valid.

In [ ]:
# The df_clean DataFrame is already created in the previous cell (QpCQ7Pwgocsm).
# Convert binary fields to 0/1.
# Missing or unrecognized amenity values are treated as 0 because they are not reported as present.
binary_mapping = {
    "true": 1, "false": 0, "t": 1, "f": 0, "yes": 1, "no": 0,
    "y": 1, "n": 0, "1": 1, "0": 0, "nan": 0, "none": 0, "": 0,
}

for col in BINARY_COLS:
    if col in df_clean.columns:
        normalized = df_clean[col].astype(str).str.strip().str.lower()
        df_clean[col] = normalized.map(binary_mapping).fillna(0).astype(int)

# Convert important numeric fields to numeric Series before creating flags.
# If a column is missing from a future data pull, use an all-NaN Series so the code still runs.
close_price = pd.to_numeric(df_clean["ClosePrice"], errors="coerce")
living_area = pd.to_numeric(df_clean["LivingArea"], errors="coerce") if "LivingArea" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
bedrooms = pd.to_numeric(df_clean["BedroomsTotal"], errors="coerce") if "BedroomsTotal" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
bathrooms = pd.to_numeric(df_clean["BathroomsTotalInteger"], errors="coerce") if "BathroomsTotalInteger" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
lot_size = pd.to_numeric(df_clean["LotSizeSquareFeet"], errors="coerce") if "LotSizeSquareFeet" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
year_built = pd.to_numeric(df_clean["YearBuilt"], errors="coerce") if "YearBuilt" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
garage_spaces = pd.to_numeric(df_clean["GarageSpaces"], errors="coerce") if "GarageSpaces" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
parking_total = pd.to_numeric(df_clean["ParkingTotal"], errors="coerce") if "ParkingTotal" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
latitude = pd.to_numeric(df_clean["Latitude"], errors="coerce") if "Latitude" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)
longitude = pd.to_numeric(df_clean["Longitude"], errors="coerce") if "Longitude" in df_clean.columns else pd.Series(np.nan, index=df_clean.index)

# Strict flags: likely data errors that should be removed before modeling.
df_clean["flag_closeprice_nonpositive"] = (close_price <= 0).fillna(False).astype(int)
df_clean["flag_closedate_before_listdate"] = (df_clean["CloseDate"] < df_clean["ListingContractDate"]).fillna(False).astype(int)

# Review flags: unusual values that might be valid for luxury or rural properties.
# These rows stay in the data, but the flags allow later error analysis by outlier type.
df_clean["flag_livingarea_extreme"] = ((living_area <= 0) | (living_area > 20000)).fillna(False).astype(int)
df_clean["flag_bedrooms_extreme"] = ((bedrooms < 0) | (bedrooms > 15)).fillna(False).astype(int)
df_clean["flag_bathrooms_extreme"] = ((bathrooms < 0) | (bathrooms > 15)).fillna(False).astype(int)
df_clean["flag_lotsize_extreme"] = ((lot_size <= 0) | (lot_size > 5_000_000)).fillna(False).astype(int)
df_clean["flag_yearbuilt_invalid"] = ((year_built < 1800) | (year_built > 2026)).fillna(False).astype(int)
df_clean["flag_garage_extreme"] = ((garage_spaces < 0) | (garage_spaces > 10)).fillna(False).astype(int)
df_clean["flag_parking_extreme"] = ((parking_total < 0) | (parking_total > 20)).fillna(False).astype(int)
df_clean["flag_latitude_outside_ca"] = ((latitude < 32) | (latitude > 42.5)).fillna(False).astype(int)
df_clean["flag_longitude_outside_ca"] = ((longitude < -125) | (longitude > -114)).fillna(False).astype(int)

# Strict issue = row should be removed.
# Review issue = row should be kept but monitored.
strict_flags = [
    "flag_closeprice_nonpositive",
    "flag_yearbuilt_invalid",
    "flag_latitude_outside_ca", "flag_longitude_outside_ca",
    "flag_closedate_before_listdate", # New strict flag
    "flag_livingarea_extreme",        # Now strict
    "flag_lotsize_extreme",           # Now strict
    "flag_bedrooms_extreme",          # Now strict
    "flag_bathrooms_extreme",         # Now strict
]
review_flags = [
    "flag_garage_extreme",
    "flag_parking_extreme",
]

df_clean["numeric_strict_issue_flag"] = df_clean[strict_flags].max(axis=1)
df_clean["numeric_review_flag"] = df_clean[review_flags].max(axis=1)

# Remove only strict numerical failures.
# Review outliers are not removed because expensive homes, large lots, or rural properties can be legitimate.
initial_len = len(df_clean)
before_strict = len(df_clean)
df_clean = df_clean[df_clean["numeric_strict_issue_flag"] == 0].copy()
removed_count_strict = before_strict - len(df_clean)
removed_pct_strict = (removed_count_strict / initial_len) * 100 if initial_len > 0 else 0
print(f"Rows removed by strict numerical rules: {removed_count_strict} ({removed_pct_strict:.2f}%)")

# Coordinates are required because the project includes geographic segmentation/enrichment.
# We remove missing coordinates instead of imputing fake locations.
if {"Latitude", "Longitude"}.issubset(df_clean.columns):
    before_coordinates = len(df_clean)
    df_clean = df_clean[df_clean["Latitude"].notna() & df_clean["Longitude"].notna()].copy()
    removed_count_coords = before_coordinates - len(df_clean)
    removed_pct_coords = (removed_count_coords / initial_len) * 100 if initial_len > 0 else 0
    print(f"Rows removed for missing coordinates: {removed_count_coords} ({removed_pct_coords:.2f}%)")
else:
    print("Latitude/Longitude not both available; coordinate filtering skipped.")

flag_cols = [c for c in df_clean.columns if c.startswith("flag_")] + ["numeric_strict_issue_flag", "numeric_review_flag"]
display(df_clean[flag_cols].sum().sort_values(ascending=False))
print("Cleaned shape:", df_clean.shape)

Rows removed by strict numerical rules: 0 (0.00%)
Rows removed for missing coordinates: 0 (0.00%)


,0
numeric_review_flag,389
flag_parking_extreme,346
flag_garage_extreme,79
flag_closedate_before_listdate,0
flag_closeprice_nonpositive,0
flag_bathrooms_extreme,0
flag_bedrooms_extreme,0
flag_livingarea_extreme,0
flag_yearbuilt_invalid,0
flag_lotsize_extreme,0


Cleaned shape: (170132, 98)


## 7. Time-Based Train/Test Split

This is not a random split. The test set is the most recent month of available closed sales. The training set is the `X` months immediately preceding that test month.

Business implication: this evaluates whether the model can support forward-looking pricing decisions under the newest market conditions.

In [ ]:
# Create a YYYY-MM close_month from CloseDate.
# The split is based on actual transaction month, not the file name.
df_clean["close_month"] = pd.to_datetime(df_clean[DATE_COL]).dt.to_period("M").astype(str)

# Define periods for test, validation, and train.
test_period = pd.Period(df_clean["close_month"].max(), freq="M")
validation_period = test_period - 1
train_end_period = validation_period - 1 # Month before validation
close_period = pd.PeriodIndex(df_clean["close_month"], freq="M")

# Show candidate training windows for train/validation/test split.
# This does not choose the optimal X by itself; model validation in Week 4 should compare performance.
split_summary_rows = []
for window in TRAINING_WINDOW_CANDIDATES:
    candidate_test_period = test_period
    candidate_validation_period = validation_period
    candidate_train_end = validation_period - 1
    candidate_train_start = candidate_train_end - (window - 1) # 'window' months for training

    candidate_train_mask = (close_period >= candidate_train_start) & (close_period <= candidate_train_end)
    candidate_validation_mask = (close_period == candidate_validation_period)
    candidate_test_mask = (close_period == candidate_test_period)

    candidate_train_rows = int(candidate_train_mask.sum())
    candidate_validation_rows = int(candidate_validation_mask.sum())
    candidate_test_rows = int(candidate_test_mask.sum())

    total_rows_in_window = candidate_train_rows + candidate_validation_rows + candidate_test_rows

    split_summary_rows.append({
        "test_month": str(candidate_test_period),
        "validation_month": str(candidate_validation_period),
        "train_start_month": str(candidate_train_start),
        "train_end_month": str(candidate_train_end),
        "training_window_months": window,
        "train_rows": candidate_train_rows,
        "validation_rows": candidate_validation_rows,
        "test_rows": candidate_test_rows,
        "excluded_rows_outside_window": len(df_clean) - total_rows_in_window,
        "candidate_status": "valid" if candidate_train_rows > 0 and candidate_validation_rows > 0 and candidate_test_rows > 0 else "invalid",
    })

split_summary = pd.DataFrame(split_summary_rows)
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / "split_window_candidate_summary.csv", index=False)

# Default split: X months for training, 1 month for validation, 1 month for test.
train_start_period = train_end_period - (TRAINING_WINDOW_MONTHS - 1)

train_mask = (close_period >= train_start_period) & (close_period <= train_end_period)
validation_mask = (close_period == validation_period)
test_mask = (close_period == test_period)

train_df = df_clean[train_mask].copy()
validation_df = df_clean[validation_mask].copy()
test_df = df_clean[test_mask].copy()

# Store split metadata for the final export.
split_info = {
    "test_month": str(test_period),
    "validation_month": str(validation_period),
    "train_start_month": str(train_start_period),
    "train_end_month": str(train_end_period),
    "training_window_months": TRAINING_WINDOW_MONTHS,
    "train_rows": len(train_df),
    "validation_rows": len(validation_df),
    "test_rows": len(test_df),
    "excluded_rows_outside_window": len(df_clean) - len(train_df) - len(validation_df) - len(test_df),
}

print(split_info)
print("Train close months:", train_df["close_month"].min(), "to", train_df["close_month"].max())
print("Validation close month:", validation_df["close_month"].unique())
print("Test close month:", test_df["close_month"].unique())

,test_month,validation_month,train_start_month,train_end_month,training_window_months,train_rows,validation_rows,test_rows,excluded_rows_outside_window,candidate_status
0,2026-05,2026-04,2026-01,2026-03,3,26469,8213,11994,123456,valid
1,2026-05,2026-04,2025-10,2026-03,6,52237,8213,11994,97688,valid
2,2026-05,2026-04,2025-07,2026-03,9,87167,8213,11994,62758,valid
3,2026-05,2026-04,2025-04,2026-03,12,122409,8213,11994,27516,valid
4,2026-05,2026-04,2025-01,2026-03,15,149925,8213,11994,0,valid
5,2026-05,2026-04,2024-12,2026-03,16,149925,8213,11994,0,valid


{'test_month': '2026-05', 'validation_month': '2026-04', 'train_start_month': '2025-04', 'train_end_month': '2026-03', 'training_window_months': 12, 'train_rows': 122409, 'validation_rows': 8213, 'test_rows': 11994, 'excluded_rows_outside_window': 27516}
Train close months: 2025-04 to 2026-03
Validation close month: ['2026-04']
Test close month: ['2026-05']


We use a **12-month training window** as the default because it balances sample size and market recency. The month immediately preceding the test month (April 2026) is used as a validation set. Shorter training windows may be more current but less stable, while longer training windows may include older market conditions that are less relevant to the May 2026 test month. The optimal value of X should be confirmed during model evaluation using R², MAPE, and MdAPE, taking into account the validation set performance.

## 8. Train-Only Imputation, Encoding, and Normalization

All preprocessing parameters (medians, modes, frequency maps, one-hot categories, scaling parameters) are fit **exclusively on the training data** to prevent data leakage. These parameters are then applied consistently to the validation and test sets.

Categorical strategy:

-   Low-cardinality categories are one-hot encoded with the first category dropped for linear-model compatibility.
-   Geographic/high-cardinality categories (and the new `PostalCode`) are frequency encoded using only the training set, preserving location signal without creating hundreds of sparse columns. This also includes the new `zip_median_price_per_sqft` aggregate feature.
-   Names, emails, addresses, and office/person identifiers are not used as model features because they do not create a stable stakeholder decision rule.

Numerical strategy:

-   Median imputation is fit on train only, then applied to validation and test sets.
-   Missingness flags are added so the model can learn whether missingness itself has signal.
-   Standardized columns are created for linear-regression baseline use; tree models can use the original numeric columns. Scaling parameters are fit on the training set and applied to validation and test sets.

In [ ]:
# Start from the time-split data.
# Important: all fitted values below must come from train only.
train_model_ready = train_df.copy()
validation_model_ready = validation_df.copy() # Add validation set
test_model_ready = test_df.copy()

# Store train-fitted preprocessing values.
# This makes it clear which medians, category maps, and scaling values were learned from train.
params = {
    "numeric_medians": {},
    "categorical_modes": {},
    "frequency_maps": {},
    "onehot_categories": {},
    "scaling": {},
}

numeric_cols = []
excluded_all_missing_numeric_cols = []

# Decide which numeric columns can actually be used.
# A numeric column that is 100% missing in train cannot be imputed meaningfully,
# so it is excluded from model-ready features and documented.
for col in KEY_NUMERIC_COLS:
    if col in train_model_ready.columns and col != TARGET_COL:
        non_missing_count = pd.to_numeric(train_model_ready[col], errors="coerce").notna().sum()
        if non_missing_count > 0:
            numeric_cols.append(col)
        else:
            excluded_all_missing_numeric_cols.append(col)

print("Numeric columns used for median imputation and scaling:", numeric_cols)
print("All-missing numeric columns excluded from model-ready features:", excluded_all_missing_numeric_cols)

# Numeric imputation:
# 1. Fit median on train.
# 2. Add a missingness flag to train, validation, and test.
# 3. Fill train, validation, and test using the train median.
for col in numeric_cols:
    train_median = pd.to_numeric(train_model_ready[col], errors="coerce").median()
    params["numeric_medians"][col] = train_median

    train_model_ready[f"{col}_was_missing"] = train_model_ready[col].isna().astype(int)
    validation_model_ready[f"{col}_was_missing"] = validation_model_ready[col].isna().astype(int) # Add validation
    test_model_ready[f"{col}_was_missing"] = test_model_ready[col].isna().astype(int)

    train_model_ready[col] = pd.to_numeric(train_model_ready[col], errors="coerce").fillna(train_median)
    validation_model_ready[col] = pd.to_numeric(validation_model_ready[col], errors="coerce").fillna(train_median) # Add validation
    test_model_ready[col] = pd.to_numeric(test_model_ready[col], errors="coerce").fillna(train_median)

# Categorical missing handling:
# Fill missing categorical values using the train mode.
# This avoids using information from the test month.
for col in FREQUENCY_CATEGORICAL_COLS + LOW_CARDINALITY_CATEGORICAL_COLS:
    if col in train_model_ready.columns:
        mode_values = train_model_ready[col].astype("string").fillna("Unknown").mode()
        mode_value = mode_values.iloc[0] if len(mode_values) else "Unknown"
        params["categorical_modes"][col] = mode_value

        train_model_ready[col] = train_model_ready[col].astype("string").fillna(mode_value).replace({"": mode_value})
        validation_model_ready[col] = validation_model_ready[col].astype("string").fillna(mode_value).replace({"": mode_value}) # Add validation
        test_model_ready[col] = test_model_ready[col].astype("string").fillna(mode_value).replace({"": mode_value})

# Frequency encoding for high-cardinality geographic categories:
# Example: City_frequency = share of training rows in that city.
# This keeps location signal without creating hundreds of dummy variables.
for col in FREQUENCY_CATEGORICAL_COLS:
    if col in train_model_ready.columns:
        frequency_map = (train_model_ready[col].astype("string").fillna("Unknown").value_counts() / len(train_model_ready)).to_dict()
        params["frequency_maps"][col] = frequency_map

        train_model_ready[f"{col}_frequency"] = train_model_ready[col].astype("string").fillna("Unknown").map(frequency_map).fillna(0)
        validation_model_ready[f"{col}_frequency"] = validation_model_ready[col].astype("string").fillna("Unknown").map(frequency_map).fillna(0) # Add validation
        test_model_ready[f"{col}_frequency"] = test_model_ready[col].astype("string").fillna("Unknown").map(frequency_map).fillna(0)

# One-hot encoding for low-cardinality categories:
# These fields have a small number of values, so dummy variables are readable and manageable.
# The first category is dropped to reduce collinearity for the linear-regression baseline.
for col in LOW_CARDINALITY_CATEGORICAL_COLS:
    if col in train_model_ready.columns:
        categories = sorted(train_model_ready[col].astype("string").fillna("Unknown").unique().tolist())
        params["onehot_categories"][col] = categories

        train_values = train_model_ready[col].astype("string").fillna("Unknown")
        validation_values = validation_model_ready[col].astype("string").fillna("Unknown") # Add validation
        test_values = test_model_ready[col].astype("string").fillna("Unknown")

        for category in categories[1:]:  # drop first category to reduce collinearity for linear baseline
            clean_category = re.sub(r"[^0-9A-Za-z]+", "_", str(category)).strip("_") or "Unknown"
            train_model_ready[f"{col}_is_{clean_category}"] = (train_values == category).astype(int)
            validation_model_ready[f"{col}_is_{clean_category}"] = (validation_values == category).astype(int) # Add validation
            test_model_ready[f"{col}_is_{clean_category}"] = (test_values == category).astype(int)

# Scaling:
# Fit mean/std on train only.
# Scaled columns are useful for the linear-regression baseline.
# Tree-based models can still use the original numeric columns.
for col in numeric_cols:
    train_values = pd.to_numeric(train_model_ready[col], errors="coerce").fillna(params["numeric_medians"].get(col, 0))
    train_mean = train_values.mean()
    train_std = train_values.std(ddof=0)
    if not train_std or np.isnan(train_std):
        train_std = 1.0

    params["scaling"][col] = {"mean": train_mean, "std": train_std}
    train_model_ready[f"{col}_scaled"] = (pd.to_numeric(train_model_ready[col], errors="coerce") - train_mean) / train_std
    validation_model_ready[f"{col}_scaled"] = (pd.to_numeric(validation_model_ready[col], errors="coerce") - train_mean) / train_std # Add validation
    test_model_ready[f"{col}_scaled"] = (pd.to_numeric(test_model_ready[col], errors="coerce") - train_mean) / train_std

print("Train model-ready shape:", train_model_ready.shape)
print("Validation model-ready shape:", validation_model_ready.shape) # Print validation shape
print("Test model-ready shape:", test_model_ready.shape)

Numeric columns used for median imputation and scaling: ['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet', 'YearBuilt', 'PropertyAge', 'GarageSpaces', 'ParkingTotal', 'Latitude', 'Longitude', 'AssociationFee', 'Stories', 'month_sin', 'month_cos']
All-missing numeric columns excluded from model-ready features: ['FireplacesTotal']
Train model-ready shape: (122409, 155)
Validation model-ready shape: (8213, 155)
Test model-ready shape: (11994, 155)


To prevent data leakage, the `zip_median_price_per_sqft` is calculated exclusively from the training set. This median value is then mapped to the validation and test sets. Any postal codes not present in the training data will be assigned the overall median `price_per_sqft` from the training set, or `0` if that's not possible.

In [ ]:
# Calculate price_per_sqft for the training data
train_df['price_per_sqft'] = train_df[TARGET_COL] / train_df['LivingArea']

# Compute median price_per_sqft for each postal code using only the training data
zip_median_price_per_sqft_map = train_df.groupby('PostalCode')['price_per_sqft'].median().to_dict()

# Get the overall median price per sqft from the training data to handle unseen postal codes
overall_median_price_per_sqft = train_df['price_per_sqft'].median()

# Map the median price per sqft to the train, validation, and test sets
train_df['zip_median_price_per_sqft'] = train_df['PostalCode'].map(zip_median_price_per_sqft_map).fillna(overall_median_price_per_sqft)
validation_df['zip_median_price_per_sqft'] = validation_df['PostalCode'].map(zip_median_price_per_sqft_map).fillna(overall_median_price_per_sqft)
test_df['zip_median_price_per_sqft'] = test_df['PostalCode'].map(zip_median_price_per_sqft_map).fillna(overall_median_price_per_sqft)

# Drop the temporary 'price_per_sqft' column from train_df
train_df = train_df.drop(columns=['price_per_sqft'])

print("Engineered 'zip_median_price_per_sqft' for train, validation, and test sets.")
display(train_df[['PostalCode', 'zip_median_price_per_sqft']].head())

Engineered 'zip_median_price_per_sqft' for train, validation, and test sets.


,PostalCode,zip_median_price_per_sqft
60176,92011,785.820964
60498,93953,1157.545853
60678,92646,808.080808
60772,90290,820.804430
60848,92078,573.385252


### 9. Filter `ClosePrice` outliers using train-set only percentiles

This step calculates the 0.5th and 99.5th percentile `ClosePrice` thresholds exclusively from the training set. These fixed thresholds are then applied to filter `ClosePrice` values in the training, validation, and test sets. This ensures that the filtering criteria are derived solely from the training data, preventing data leakage from the validation or test sets.

In [ ]:
# Calculate 0.5th and 99.5th percentiles from the training set only
lower_bound = train_model_ready[TARGET_COL].quantile(0.005)
upper_bound = train_model_ready[TARGET_COL].quantile(0.995)

print(f"ClosePrice 0.5th percentile (from train set): {lower_bound:.2f}")
print(f"ClosePrice 99.5th percentile (from train set): {upper_bound:.2f}")

# Function to apply filtering and log removed rows
def filter_and_log(df_input, name, lower, upper):
    initial_len = len(df_input)
    df_filtered = df_input[
        (df_input[TARGET_COL] >= lower) & (df_input[TARGET_COL] <= upper)
    ].copy()
    removed_count = initial_len - len(df_filtered)
    removed_pct = (removed_count / initial_len) * 100 if initial_len > 0 else 0
    print(f"Rows removed from {name} set by ClosePrice thresholds: {removed_count} ({removed_pct:.2f}%)")
    return df_filtered

# Apply filtering to all datasets
train_model_ready = filter_and_log(train_model_ready, "Training", lower_bound, upper_bound)
validation_model_ready = filter_and_log(validation_model_ready, "Validation", lower_bound, upper_bound)
test_model_ready = filter_and_log(test_model_ready, "Test", lower_bound, upper_bound)

print("\nShapes after ClosePrice filtering:")
print("Train model-ready shape:", train_model_ready.shape)
print("Validation model-ready shape:", validation_model_ready.shape)
print("Test model-ready shape:", test_model_ready.shape)

ClosePrice 0.5th percentile (from train set): 189000.00
ClosePrice 99.5th percentile (from train set): 8700000.00
Rows removed from Training set by ClosePrice thresholds: 1223 (1.00%)
Rows removed from Validation set by ClosePrice thresholds: 62 (0.75%)
Rows removed from Test set by ClosePrice thresholds: 113 (0.94%)

Shapes after ClosePrice filtering:
Train model-ready shape: (121186, 155)
Validation model-ready shape: (8151, 155)
Test model-ready shape: (11881, 155)


In [ ]:
# File names are explicit so the team can tell which artifact is for auditing
# and which artifact is ready for modeling.
quality_cleaned_csv = OUTPUT_DIR / "crmls_sfr_quality_cleaned_202501_202605.csv"
model_ready_combined_csv = OUTPUT_DIR / f"crmls_sfr_model_ready_X{TRAINING_WINDOW_MONTHS}_train_validation_test_combined.csv" # Updated name
train_csv = OUTPUT_DIR / f"crmls_sfr_train_X{TRAINING_WINDOW_MONTHS}_{split_info['train_start_month']}_to_{split_info['train_end_month']}.csv"
validation_csv = OUTPUT_DIR / f"crmls_sfr_validation_{split_info['validation_month']}.csv" # New validation csv
test_csv = OUTPUT_DIR / f"crmls_sfr_test_{split_info['test_month']}.csv"
metadata_csv = OUTPUT_DIR / "preprocessing_metadata.csv"

# Add a split label before combining train, validation, and test.
# This combined file is convenient for Tableau checks or downstream modeling scripts.
train_model_ready["split"] = "train"
validation_model_ready["split"] = "validation" # Add split label for validation
test_model_ready["split"] = "test"
model_ready_combined = pd.concat([train_model_ready, validation_model_ready, test_model_ready], ignore_index=True, sort=False) # Concat all three

# Missing-value checks:
# quality_cleaned_missing shows what is still missing in the audit dataset.
# model_ready_missing verifies that engineered model-ready columns have no missing values.
quality_cleaned_feature_check = [c for c in KEY_NUMERIC_COLS + FREQUENCY_CATEGORICAL_COLS + LOW_CARDINALITY_CATEGORICAL_COLS if c in df_clean.columns]
model_ready_feature_check = [
    c for c in model_ready_combined.columns
    if c.endswith("_scaled") or c.endswith("_frequency") or c.endswith("_was_missing") or "_is_" in c
]

quality_cleaned_missing = df_clean[quality_cleaned_feature_check].isna().sum().sort_values(ascending=False)
model_ready_missing = model_ready_combined[model_ready_feature_check].isna().sum().sort_values(ascending=False)

quality_cleaned_missing.to_csv(OUTPUT_DIR / "quality_cleaned_missing_check.csv")
model_ready_missing.to_csv(OUTPUT_DIR / "model_ready_missing_check.csv")

# Export the actual deliverables.
df_clean.to_csv(quality_cleaned_csv, index=False)
model_ready_combined.to_csv(model_ready_combined_csv, index=False)
train_model_ready.to_csv(train_csv, index=False)
validation_model_ready.to_csv(validation_csv, index=False) # Export validation csv
test_model_ready.to_csv(test_csv, index=False)

# Metadata is a compact audit trail for reviewers.
metadata = pd.DataFrame([
    {"item": "target", "value": TARGET_COL},
    {"item": "filter", "value": 'PropertyType == "Residential" and PropertySubType == "SingleFamilyResidence"'},
    {"item": "test_month", "value": split_info["test_month"]},
    {"item": "validation_month", "value": split_info["validation_month"]}, # Add validation month
    {"item": "train_window_months", "value": split_info["training_window_months"]},
    {"item": "train_months", "value": f"{split_info['train_start_month']} to {split_info['train_end_month']}"},
    {"item": "train_rows", "value": split_info["train_rows"]},
    {"item": "validation_rows", "value": split_info["validation_rows"]}, # Add validation rows
    {"item": "test_rows", "value": split_info["test_rows"]},
    {"item": "excluded_rows_outside_window", "value": split_info["excluded_rows_outside_window"]},
    {"item": "all_missing_numeric_excluded_from_model_ready", "value": ", ".join(excluded_all_missing_numeric_cols) if excluded_all_missing_numeric_cols else "none"},
    {"item": "model_ready_engineered_feature_max_missing", "value": int(model_ready_missing.max()) if len(model_ready_missing) else 0},
    {"item": "quality_cleaned_csv", "value": quality_cleaned_csv.name},
    {"item": "model_ready_combined_csv", "value": model_ready_combined_csv.name},
    {"item": "train_csv", "value": train_csv.name}, # Add train_csv to metadata
    {"item": "validation_csv", "value": validation_csv.name}, # Add validation_csv to metadata
    {"item": "test_csv", "value": test_csv.name}, # Add test_csv to metadata
    {"item": "decision_implication", "value": "Model evaluation simulates pricing decisions on the newest available closed-sale month using only immediately prior months for training and the month before test for validation."}
])
metadata.to_csv(metadata_csv, index=False)

print("Wrote:")
print(quality_cleaned_csv)
print(model_ready_combined_csv)
print(train_csv)
print(validation_csv) # Print validation csv
print(test_csv)
print(metadata_csv)

Wrote:
/content/drive/MyDrive/IDX Exchange/outputs/crmls_sfr_quality_cleaned_202501_202605.csv
/content/drive/MyDrive/IDX Exchange/outputs/crmls_sfr_model_ready_X12_train_validation_test_combined.csv
/content/drive/MyDrive/IDX Exchange/outputs/crmls_sfr_train_X12_2025-04_to_2026-03.csv
/content/drive/MyDrive/IDX Exchange/outputs/crmls_sfr_validation_2026-04.csv
/content/drive/MyDrive/IDX Exchange/outputs/crmls_sfr_test_2026-05.csv
/content/drive/MyDrive/IDX Exchange/outputs/preprocessing_metadata.csv


## Limitations

-   The notebook creates a defensible split and preprocessing artifacts; it does not claim the optimal value of `X` without model validation.
-   Frequency encoding captures category prevalence, not causal location effects.
-   Missing coordinates are removed to support geographic decision use cases, which may underrepresent records with poor geocoding.
-   Review outliers remain in the data and must be checked during model error analysis by price segment.

## Decision Implication

The exported train/validation/test files let the team evaluate pricing models on the newest available market month while training only on prior months and validating on the month immediately before the test set. That is the right evaluation structure for listing-pricing decisions because it mimics how the model would be used prospectively.

## 10. Linear Regression Model Training and Evaluation

This section trains a simple linear regression model on the preprocessed training data. The model's performance will be evaluated on both the validation and test sets using metrics such as Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE), and R-squared (R²).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define target and features
TARGET_COL = 'ClosePrice'

# Select features that are ready for modeling (scaled numerical, frequency encoded, one-hot encoded, binary flags)
feature_columns = [
    c for c in model_ready_combined.columns
    if c.endswith('_scaled') or c.endswith('_frequency') or c.endswith('_was_missing') or '_is_' in c or c in BINARY_COLS or c == 'zip_median_price_per_sqft'
]

# Exclude the target column and any non-feature columns from the feature list
if TARGET_COL in feature_columns:
    feature_columns.pop(feature_columns.index(TARGET_COL))

# Filter out any other columns that are not meant to be features
# This might include original columns for which scaled/encoded versions exist, or identifiers
excluded_from_features = [
    'source_month', 'close_month', 'PropertyType', 'PropertySubType',
    'Unnamed: 0', 'split', 'PropertyAge', 'YearBuilt', 'LivingArea', 'BedroomsTotal',
    'BathroomsTotalInteger', 'LotSizeSquareFeet', 'GarageSpaces', 'ParkingTotal',
    'Latitude', 'Longitude', 'AssociationFee', 'FireplacesTotal', 'Stories',
    'month_sin', 'month_cos' # Original numerical features, as scaled versions are used
]
feature_columns = [f for f in feature_columns if f not in excluded_from_features]

X_train = train_model_ready[feature_columns]
y_train = train_model_ready[TARGET_COL]

X_val = validation_model_ready[feature_columns]
y_val = validation_model_ready[TARGET_COL]

X_test = test_model_ready[feature_columns]
y_test = test_model_ready[TARGET_COL]

print(f"Number of features: {len(feature_columns)}")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Initialize and train the Linear Regression model
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

print("Linear Regression model trained successfully.")

In [ ]:
# Make predictions on the validation and test sets
y_val_pred = linear_model.predict(X_val)
y_test_pred = linear_model.predict(X_test)

# Evaluate model performance on validation set
print("\nValidation Set Metrics:")
print(f"MAE: {mean_absolute_error(y_val, y_val_pred):,.2f}")
print(f"MSE: {mean_squared_error(y_val, y_val_pred):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_val, y_val_pred)):,.2f}")
print(f"R-squared: {r2_score(y_val, y_val_pred):.4f}")

# Evaluate model performance on test set
print("\nTest Set Metrics:")
print(f"MAE: {mean_absolute_error(y_test, y_test_pred):,.2f}")
print(f"MSE: {mean_squared_error(y_test, y_test_pred):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):,.2f}")
print(f"R-squared: {r2_score(y_test, y_test_pred):.4f}")

### Residual Analysis

Visualizing the residuals can help identify patterns or issues with the model's predictions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a DataFrame for residuals
residuals_df_val = pd.DataFrame({
    'Actual': y_val,
    'Predicted': y_val_pred,
    'Residuals': y_val - y_val_pred
})

residuals_df_test = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_test_pred,
    'Residuals': y_test - y_test_pred
})

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.histplot(residuals_df_val['Residuals'], kde=True)
plt.title('Validation Set Residuals Distribution')
plt.xlabel('Residuals')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
sns.histplot(residuals_df_test['Residuals'], kde=True)
plt.title('Test Set Residuals Distribution')
plt.xlabel('Residuals')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.scatterplot(x='Predicted', y='Residuals', data=residuals_df_val, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Validation Set: Residuals vs. Predicted Values')
plt.xlabel('Predicted ClosePrice')
plt.ylabel('Residuals')

plt.subplot(1, 2, 2)
sns.scatterplot(x='Predicted', y='Residuals', data=residuals_df_test, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.title('Test Set: Residuals vs. Predicted Values')
plt.xlabel('Predicted ClosePrice')
plt.ylabel('Residuals')

plt.tight_layout()
plt.show()